# Colab Training: Anomaly Detection

Two shell commands to run the full pipeline. Edit the variables below, then **Run All**.

1. Mount Drive + load Kaggle credentials
2. **Setup** — clone, install deps, update config, pull dataset
3. **Train** — split, train, evaluate, export

In [ ]:
import os

# ──────────────── EDIT THESE ────────────────
GITHUB_REPO_URL = "https://github.com/AakashBhat1/collab_training_anomaly_detection.git"
GITHUB_BRANCH   = "main"
KAGGLE_DATASET  = "webadvisor/real-time-anomaly-detection-in-cctv-surveillance"
# ─────────────────────────────────────────────

REPO_DIR = "/content/intruder_detection_system"

# Mount Google Drive (for checkpoint backup/resume)
from google.colab import drive
drive.mount("/content/drive")

# Load Kaggle credentials from Colab Secrets or env
kaggle_user = os.environ.get("KAGGLE_USERNAME", "")
kaggle_key  = os.environ.get("KAGGLE_KEY", "")
if not kaggle_user or not kaggle_key:
    try:
        from google.colab import userdata
        kaggle_user = kaggle_user or userdata.get("KAGGLE_USERNAME")
        kaggle_key  = kaggle_key  or userdata.get("KAGGLE_KEY")
    except Exception:
        pass
if kaggle_user and kaggle_key:
    os.environ["KAGGLE_USERNAME"] = kaggle_user
    os.environ["KAGGLE_KEY"]      = kaggle_key
    print("Kaggle credentials loaded.")
else:
    print("WARNING: Kaggle credentials not found. Set them in Colab Secrets.")

In [ ]:
# Step 1: Setup — clone repo, install deps, update config, pull Kaggle data
# (colab_setup.sh handles clone internally, but we need the script first)
import subprocess, sys

if not __import__("pathlib").Path(f"{REPO_DIR}/.git").exists():
    subprocess.run(
        ["git", "clone", "--branch", GITHUB_BRANCH, "--single-branch",
         GITHUB_REPO_URL, REPO_DIR],
        check=True,
    )
    print(f"Cloned -> {REPO_DIR}")
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull"], check=True)
    print(f"Pulled latest in {REPO_DIR}")

# Now run the setup script from the cloned repo
!bash {REPO_DIR}/collab_scripts/scripts/colab_setup.sh \
    {GITHUB_REPO_URL} \
    --branch {GITHUB_BRANCH} \
    --repo-dir {REPO_DIR} \
    --kaggle-dataset {KAGGLE_DATASET} \
    --kaggle-clean

In [ ]:
# Step 2: Train — split dataset, train model, evaluate, export to ONNX/OpenVINO
!bash {REPO_DIR}/collab_scripts/scripts/colab_run_training.sh {REPO_DIR}